# 28. Stored Procedures

*Module: 7 — Advanced server · Companion summary: `lectures/28-stored-procedures.md` · SQL engine: SQLite (in-memory)*

## What you will study

- explain what a stored procedure is and why engines love them
- read the course's MySQL update_sal procedure
- emulate one with a Python function — SQLite's reality

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 📗 What is a stored procedure?

A **named, saved procedure of SQL stored *in the database***. Called instead of
sending statements one by one. Benefits:

- ✅ Encapsulation — logic lives inside the db; applications just call it
- ✅ Performance — parsed/optimized once (and no per-statement round trips)
- ✅ Security — you can grant EXECUTE without raw-table permissions
- ✅ Reduced network traffic

**SQLite reality check:** SQLite has **no stored procedure language** — so the course's
MySQL procedure is studied as-is, and its behavior is emulated with a Python function
calling parameterized SQL (which is what nb22 trained you for).

## 📕 The course's MySQL procedure (non-runnable here, learnable anywhere)

```sql
DELIMITER //
CREATE PROCEDURE update_sal (IN emp_num CHAR(6), IN rating INT)
BEGIN
    IF rating = 1 THEN
        UPDATE employees SET salary = salary * 1.10
        WHERE emp_num = emp_num;
    ELSE
        UPDATE employees SET salary = salary * 1.05
        WHERE emp_num = emp_num;
    END IF;
END //
DELIMITER ;
CALL update_sal('E1234', 1);   -- how the app invokes it
```

## 🧪 Build the same behavior in Python + SQLite

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)


In [ ]:
q("CREATE TABLE employees (emp_num TEXT PRIMARY KEY, name TEXT, salary REAL)")
q("""
INSERT INTO employees VALUES
    ('E1234','Raul',100000), ('E2345','Rav',80000), ('E3456','Ana',60000)
""")
q("SELECT * FROM employees")

In [ ]:
def update_sal(conn, emp_num, rating):
    """Emulates the stored procedure: +10% for rating 1, +5% otherwise."""
    factor = 1.10 if rating == 1 else 1.05
    with conn:
        cur = conn.execute(
            "UPDATE employees SET salary = ROUND(salary * ?, 2) WHERE emp_num = ?",
            (factor, emp_num))
        return cur.rowcount

print(update_sal(conn, 'E1234', 1))   # like CALL update_sal('E1234', 1)
q("SELECT * FROM employees")

In [ ]:
update_sal(conn, 'E2345', 1)
update_sal(conn, 'E3456', 2)   # not rating 1 → only 5%
q("SELECT emp_num, name, salary FROM employees ORDER BY salary DESC")

## 💡 Tips & tricks

- Parameterize, never f-string: the procedure body above uses `?` placeholders exactly
  like a stored proc uses parameters.
- If you later meet DB2/Postgres (CREATE PROCEDURE / LANGUAGE SQL PL/pgSQL), the
  CALL semantics match this emulation.
- In analytics pipelines, a Python function + parameterized SQL is often *simpler* than
  a server-side proc — know both.

## 🎯 Your turn

1. Add a 3-rating rule of your own (rating 3 → −10%) to update_sal.
2. Build and call your own 'procedure' `hire_employee` that INSERTs a new employee.

In [ ]:
# 💡 Solution
def update_sal2(conn, emp_num, rating):
    factor = {1: 1.10, 2: 1.05, 3: 0.90}[rating]
    with conn:
        conn.execute("UPDATE employees SET salary = ROUND(salary * ?, 2) WHERE emp_num = ?",
                     (factor, emp_num))
update_sal2(conn, 'E3456', 3)
q("SELECT * FROM employees WHERE emp_num = 'E3456'")

def hire_employee(conn, num, name, salary):
    with conn:
        conn.execute("INSERT INTO employees VALUES (?, ?, ?)", (num, name, salary))
hire_employee(conn, 'E9999', 'You', 72000)
q("SELECT * FROM employees WHERE emp_num = 'E9999'")